# 03 - Ablations

Two knobs: how large the replay buffer is, and how often consolidation happens.
Both are full runs per variant, so this is the most expensive notebook here.
Run one grid per session and let the checkpoints accumulate.

The CPU pipeline runs the same grids far more cheaply and is the right place to
sanity-check a grid before spending GPU time on it:

    python scripts/run_ablations.py --grid buffer

## Environment

In [ ]:
import os, sys, subprocess

REPO = "/kaggle/working/skill-library"
if not os.path.exists(REPO):
    # On Kaggle, add the repo as a dataset or clone it here.
    REPO = os.getcwd()
sys.path.insert(0, REPO)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft>=0.11", "transformers>=4.40", "accelerate"], check=False)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name}  {total:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")

## Configuration

In [ ]:
from baselines.gpu_runner import GPURun, load_config
from sleep.lora import load_base_model

config = load_config("configs/default.yaml")
model, tokenizer = load_base_model(
    config["model"]["name"], config["model"]["device"], config["model"]["dtype"]
)

# Ablations are the expensive part of the project. Each variant is a full run,
# so pick one grid per session rather than trying to do all three.
GRID = "buffer"     # "buffer" | "frequency"
NIGHTS_THIS_SESSION = 3

## Run the grid

In [ ]:
import copy, torch
from pathlib import Path

GRIDS = {
    "buffer": [("buffer_0", {"max_total": 0, "per_skill": 0, "enabled": False}),
               ("buffer_6", {"max_total": 6, "per_skill": 1}),
               ("buffer_12", {"max_total": 12, "per_skill": 1}),
               ("buffer_24", {"max_total": 24, "per_skill": 2})],
    "frequency": [("K_1", {}), ("K_3", {}), ("K_5", {})],
}
K_FOR = {"K_1": 1, "K_3": 3, "K_5": 5}

for variant, replay in GRIDS[GRID]:
    variant_config = copy.deepcopy(config)
    variant_config["run"]["run_id"] = f"gpu_ablation_{GRID}"
    variant_config["sleep"]["replay"].update(replay)
    if variant in K_FOR:
        variant_config["sleep"]["every_k_tasks"] = K_FOR[variant]

    run = GPURun(variant_config, method="ours", model=model, tokenizer=tokenizer)
    run.root = Path(variant_config["run"]["root"]) / variant_config["run"]["run_id"] / variant
    resumed = run.resume()
    print(f"[{variant}] resumed at {resumed}, {len(run.library)} skills")
    run.run(max_nights=NIGHTS_THIS_SESSION)
    print(f"[{variant}] night {run.consolidator.night_count}, {len(run.library)}/15")

    del run
    torch.cuda.empty_cache()

## Collect

In [ ]:
from pathlib import Path
from analysis.collect import summary_table, write_collected
import json

root = Path(config["run"]["root"]) / f"gpu_ablation_{GRID}"
variants = [v for v, _ in GRIDS[GRID]]
path = write_collected(root, variants, run_id=f"gpu_ablation_{GRID}")
print(summary_table(json.loads(path.read_text())))